# LegalLens — Fine-Tuning InLegalBERT on CUAD for 15 Clause Categories

This notebook trains `law-ai/InLegalBERT` on the Contract Understanding Atticus Dataset (CUAD) to classify contract clauses into the 15 most consequential legal categories for the LegalLens platform.

### Core 15 Categories:
1. `Termination`
2. `Indemnification`
3. `Limitation_of_Liability`
4. `Confidentiality`
5. `Non_Compete`
6. `Governing_Law`
7. `Dispute_Resolution`
8. `Intellectual_Property`
9. `Payment_Terms`
10. `Term_and_Renewal`
11. `Warranties`
12. `Exclusivity_and_Non_Solicit`
13. `Severability`
14. `Force_Majeure`
15. `Assignment`

In [17]:
!nvidia-smi

Sat Sep 12 11:36:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             27W /   70W |     577MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q transformers datasets evaluate accelerate scikit-learn torch

Sat Sep 12 11:36:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             27W /   70W |     577MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
# 2. Imports and Reproducibility
import os
import json
import torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [20]:
# 3. Load Canonical 15 Core Categories from backend/ml/categories.py
import os
import sys
import subprocess
from pathlib import Path

def load_canonical_categories():
    """
    Loads the canonical 15 clause categories defined in LegalLens backend/ml/categories.py.
    Supports local execution as well as Google Colab environments (via repo clone or raw GitHub fetch).
    """
    # 1. Check local repository relative paths
    candidate_paths = [
        Path("backend/ml/categories.py"),
        Path("../backend/ml/categories.py"),
        Path("/content/LegalLens/backend/ml/categories.py"),
    ]
    for p in candidate_paths:
        if p.is_file():
            sys.path.insert(0, str(p.parent.parent.parent.resolve()))
            try:
                from backend.ml.categories import CATEGORIES
                return list(CATEGORIES)
            except Exception:
                namespace = {}
                with open(p, "r", encoding="utf-8") as f:
                    exec(f.read(), namespace)
                if "CATEGORIES" in namespace:
                    return list(namespace["CATEGORIES"])

    # 2. In Colab, clone the repo if not present
    repo_url = os.getenv("LEGALLENS_REPO_URL", "https://github.com/athulvinod06/LegalLens.git")
    print(f"Categories file not found locally. Attempting to clone from {repo_url}...")
    try:
        subprocess.run(["git", "clone", "--depth", "1", repo_url, "/content/LegalLens"], check=True)
        colab_path = Path("/content/LegalLens/backend/ml/categories.py")
        if colab_path.is_file():
            sys.path.insert(0, "/content/LegalLens")
            from backend.ml.categories import CATEGORIES
            return list(CATEGORIES)
    except Exception as e:
        print(f"Git clone attempt failed: {e}")

    # 3. Fallback: Download raw backend/ml/categories.py from GitHub
    raw_url = os.getenv(
        "CATEGORIES_RAW_URL",
        "https://raw.githubusercontent.com/athulvinod06/LegalLens/main/backend/ml/categories.py"
    )
    try:
        import urllib.request
        print(f"Attempting to download categories.py from {raw_url}...")
        with urllib.request.urlopen(raw_url, timeout=10) as resp:
            content = resp.read().decode("utf-8")
            namespace = {}
            exec(content, namespace)
            if "CATEGORIES" in namespace:
                return list(namespace["CATEGORIES"])
    except Exception as e:
        print(f"Raw download attempt failed: {e}")

    raise RuntimeError(
        "CRITICAL ERROR: Failed to load canonical CATEGORIES from backend/ml/categories.py. "
        "Ensure the LegalLens repository is accessible or cloned into the execution environment."
    )

CATEGORIES = load_canonical_categories()

# Sanity check on loaded categories: must have exactly 15 unique, non-empty strings
if not isinstance(CATEGORIES, list):
    raise AssertionError(f"CATEGORIES must be a list, got {type(CATEGORIES)}")
if len(CATEGORIES) != 15:
    raise AssertionError(
        f"CATEGORIES must contain exactly 15 entries as specified in LegalLens schema, "
        f"but found {len(CATEGORIES)}: {CATEGORIES}"
    )
for idx, cat in enumerate(CATEGORIES):
    if not isinstance(cat, str) or not cat.strip():
        raise AssertionError(f"CATEGORIES[{idx}] is invalid or empty: {cat!r}")
if len(set(CATEGORIES)) != 15:
    raise AssertionError(f"CATEGORIES contains duplicate entries: {CATEGORIES}")

label2id = {cat: idx for idx, cat in enumerate(CATEGORIES)}
id2label = {idx: cat for cat, idx in label2id.items()}

print(f"Successfully loaded and verified {len(CATEGORIES)} canonical categories:")
for idx, cat in enumerate(CATEGORIES):
    print(f"  {idx:2d}: {cat}")


Successfully loaded and verified 15 canonical categories:
   0: Termination
   1: Indemnification
   2: Limitation_of_Liability
   3: Confidentiality
   4: Non_Compete
   5: Governing_Law
   6: Dispute_Resolution
   7: Intellectual_Property
   8: Payment_Terms
   9: Term_and_Renewal
  10: Warranties
  11: Exclusivity_and_Non_Solicit
  12: Severability
  13: Force_Majeure
  14: Assignment


In [21]:
# 4. Load CUAD Dataset & Filter for Target Categories
print("Loading CUAD dataset...")
# CUAD can be loaded via huggingface datasets:
try:
    cuad_raw = load_dataset("TheAtticusProject/cuad", split="train")
    print("Loaded CUAD successfully from Hugging Face.")
except Exception as e:
    print(f"Direct load error: {e}. If offline or in standalone mode, provide local path to CUAD_v1.json.")

# Preprocessing helper to map CUAD question / category titles to our 15 canonical classes
CUAD_CATEGORY_MAP = {
    "Termination For Convenience": "Termination",
    "Notice To Terminate Renewal": "Termination",
    "Indebtedness": "Payment_Terms",
    "Cap On Liability": "Limitation_of_Liability",
    "Liquidated Damages": "Limitation_of_Liability",
    "Non-Compete": "Non_Compete",
    "Exclusivity": "Exclusivity_and_Non_Solicit",
    "No-Solicit Of Customers": "Exclusivity_and_Non_Solicit",
    "No-Solicit Of Employees": "Exclusivity_and_Non_Solicit",
    "Governing Law": "Governing_Law",
    "Dispute Resolution": "Dispute_Resolution",
    "IP Ownership Assignment": "Intellectual_Property",
    "License grant": "Intellectual_Property",
    "Confidentiality": "Confidentiality",
    "Non-Disclosure": "Confidentiality",
    "Post-Termination Services": "Termination",
    "Audit Rights": "Payment_Terms",
    "Uncapped Liability": "Limitation_of_Liability",
    "Warranty Duration": "Warranties",
    "Severability": "Severability",
    "Force Majeure": "Force_Majeure",
    "Anti-Assignment": "Assignment",
    "Renewal Term": "Term_and_Renewal",
    "Initial Term": "Term_and_Renewal"
}

Loading CUAD dataset...


Resolving data files:   0%|          | 0/714 [00:00<?, ?it/s]

Loaded CUAD successfully from Hugging Face.


In [22]:
# 5. Tokenizer & InLegalBERT Model Setup
BASE_MODEL = "law-ai/InLegalBERT"
print(f"Initializing tokenizer and base model from {BASE_MODEL}...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(CATEGORIES),
    id2label=id2label,
    label2id=label2id
)
model.to(DEVICE)
print("Model loaded.")

Initializing tokenizer and base model from law-ai/InLegalBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were n

Model loaded.


In [23]:
# 6. Evaluation Metrics: Precision, Recall, Macro-F1
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="macro", zero_division=0)["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="macro", zero_division=0)["recall"]
    return {
        "accuracy": acc,
        "macro_f1": f1,
        "precision": prec,
        "recall": rec
    }

In [24]:
# 7. Training Arguments & Trainer Setup
training_args = TrainingArguments(
    output_dir="./inlegalbert_cuad_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    seed=SEED
)

print("Training configuration ready.")

Training configuration ready.


In [35]:
!pip install -q pdfplumber

In [38]:
import importlib
import datasets.config
importlib.reload(datasets.config)

<module 'datasets.config' from '/usr/local/lib/python3.13/dist-packages/datasets/config.py'>

In [40]:
print(cuad_raw.column_names)
print(cuad_raw[0])

['pdf']
{'pdf': <pdfplumber.pdf.PDF object at 0x7ee45bfa76a0>}


In [39]:
# 7.5. Construct Labeled Dataset, Tokenize & Fine-Tune Model

def get_clause_category(item):
    """Maps CUAD category / question to one of the 15 canonical categories via CUAD_CATEGORY_MAP."""
    cat_id = item.get("id", "").split("__")[-1].strip() if "__" in item.get("id", "") else ""

    # Direct match in CUAD_CATEGORY_MAP
    if cat_id in CUAD_CATEGORY_MAP:
        return CUAD_CATEGORY_MAP[cat_id]

    # Case-insensitive match on ID category
    for k, v in CUAD_CATEGORY_MAP.items():
        if k.lower() == cat_id.lower():
            return v

    # Normalized match (handles variations like 'Notice Period To Terminate Renewal' vs 'Notice To Terminate Renewal')
    norm_cat = " ".join(cat_id.lower().replace("period", "").replace("-", " ").split())
    for k, v in CUAD_CATEGORY_MAP.items():
        norm_k = " ".join(k.lower().replace("period", "").replace("-", " ").split())
        if norm_k == norm_cat:
            return v

    # Fallback to question text matching
    q_text = item.get("question", "")
    for k, v in CUAD_CATEGORY_MAP.items():
        if f'"{k}"' in q_text or k.lower() in q_text.lower():
            return v

    return None

# 1. Filter to positive/answered spans and map each to target canonical categories
labeled_examples = []
for item in cuad_raw:
    category = get_clause_category(item)
    if not category or category not in label2id:
        continue

    answers = item.get("answers", {})
    if isinstance(answers, dict):
        answer_texts = answers.get("text", [])
    elif isinstance(answers, list):
        answer_texts = [a.get("text", "") for a in answers if isinstance(a, dict)]
    else:
        answer_texts = []

    for span in answer_texts:
        if isinstance(span, str) and span.strip():
            labeled_examples.append({
                "text": span.strip(),
                "label": label2id[category]
            })

print(f"Extracted {len(labeled_examples)} positive clause annotations mapped to canonical categories.")
raw_dataset = Dataset.from_list(labeled_examples)

# 2. 80/20 train/eval split using existing SEED
split_dataset = raw_dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Dataset split: {len(train_dataset)} train samples, {len(eval_dataset)} eval samples.")

# 3. Tokenize using existing tokenizer with truncation and max_length=512
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_eval = eval_dataset.map(tokenize_fn, batched=True)

# 4. Instantiate Trainer using existing variables
if "data_collator" not in globals():
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 5. Execute training
print("Starting InLegalBERT fine-tuning...")
trainer.train()


Extracted 0 positive clause annotations mapped to canonical categories.
Dataset split: 0 train samples, 0 eval samples.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [37]:
%whos


Variable                             Type                             Data/Info
-------------------------------------------------------------------------------
AutoModelForSequenceClassification   type                             <class 'transformers.mode<...>rSequenceClassification'>
AutoTokenizer                        type                             <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
BASE_MODEL                           str                              law-ai/InLegalBERT
CATEGORIES                           list                             n=15
CUAD_CATEGORY_MAP                    dict                             n=24
DEVICE                               str                              cuda
DataCollatorWithPadding              type                             <class 'transformers.data<...>DataCollatorWithPadding'>
Dataset                              type                             <class 'datasets.arrow_dataset.Dataset'>
OUTPUT_MODEL_DIR                     

In [25]:
# 8. Save Fine-Tuned Model Checkpoint for LegalLens Backend
OUTPUT_MODEL_DIR = "./legallens_inlegalbert_model"
# Once trainer.train() completes:
trainer.save_model(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)

# Save label mapping
with open("label_mapping.json", "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)
print(f"Label mapping saved. Ready to download and place in backend/ml/checkpoints/")

NameError: name 'trainer' is not defined

In [ ]:
import os
print(os.listdir(OUTPUT_MODEL_DIR))

In [ ]:
print(trainer)